<div style='background-color:#002147; padding:20px; border-radius:8px'>
<h1 style='color:white;'>Cálculo Numérico</h1>
<h3 style='color:white;'>Resolução de sistemas lineares: Fatoração LU</h3>
</div>

### 🎯 Objetivos de aprendizagem

Ao final deste material, você deverá ser capaz de:
- Compreender o conceito de fatoração LU
- Implementar o método da fatoração LU
- Resolver sistemas lineares utilizando a fatoração LU
- Compreender a importância do pivoteamento parcial para a estabilidade numérica

## $ \S 0 $ Preparação do ambiente

Execute a célula abaixo para carregar as bibliotecas utilizadas neste Jupyter Notebook. 💻

Usaremos `numpy` para manipular matrizes e vetores ao longo das atividades.

In [1]:
import numpy as np

**🔹 Funções auxiliares: visualização de matrizes e vetores**

Antes de implementar os métodos numéricos, criamos funções auxiliares para exibir as matrizes e acompanhar a execução passo a passo.

In [2]:
def imprimir_matriz(M, titulo="Matriz"):
    """
    Imprime uma matriz com identificação das linhas.
    """
    M = M.astype(float)
    n_linhas, n_colunas = M.shape
    largura = 12*n_colunas

    print(f"\n🔹 {titulo}")
    print("   ┌" + " " * largura + "┐")

    for i in range(n_linhas):
        linha = f"L{i+1} "
        for j in range(n_colunas):
            linha += f"{M[i, j]:10.3e} "
        #linha += "│"
        print(linha)

    print("   └" + " " * largura + "┘")


def imprimir_vetor(v, titulo="Vetor"):
    """
    Imprime um vetor coluna.
    """
    v = v.astype(float)
    print(f"\n🔹 {titulo}")
    for i, valor in enumerate(v):
        print(f"[{i+1}] {valor:10.3e}")


def imprimir_fatoracao(L, U, titulo="Fatoração atual"):
    """
    Exibe as matrizes L e U de forma organizada.
    """
    imprimir_matriz(L, f"{titulo}: matriz L")
    imprimir_matriz(U, f"{titulo}: matriz U")


def imprimir_fatoracao_pivotada(P, L, U, titulo="Fatoração atual"):
    """
    Exibe as matrizes P, L e U de forma organizada.
    """
    imprimir_matriz(P, f"{titulo}: matriz P")
    imprimir_matriz(L, f"{titulo}: matriz L")
    imprimir_matriz(U, f"{titulo}: matriz U")

<div style="background-color: #fff3b0; padding: 12px; border-radius: 6px;">

⚠️ **Atenção:**
- Python → índices começam em 0.
- Na notação matemática, é comum contar linhas e colunas a partir de 1.
</div>

## $ \S 1 $ Fatoração LU

Seguindo a apresentação clássica do livro de Ruggiero e Lopes, a fatoração LU pode ser entendida como uma forma matricial de organizar a eliminação de Gauss.

Seja $A$ uma matriz quadrada não singular. Quando o escalonamento de $A$ pode ser feito sem troca de linhas, podemos escrever

$$
A = LU,
$$

onde:

- $L$ é triangular inferior com diagonal principal igual a 1;
- $U$ é triangular superior.

Nessa decomposição:

- a matriz $U$ é a matriz resultante do escalonamento;
- os multiplicadores usados na eliminação ficam armazenados em $L$.

<div style='background-color:#f0f0f0; padding:20px; border-radius:8px'>

🎯 A principal vantagem da fatoração LU aparece quando precisamos resolver vários sistemas com a mesma matriz $A$ e vetores independentes diferentes.

Nesse caso, a fatoração é feita uma única vez.
</div>

## $ \S 2 $ Resolução via sistemas triangulares

Se $A = LU$, então resolver o sistema

$$
Ax = b
$$

equivale a resolver o par de sistemas

$$
\begin{cases}
Ly = b \\
Ux = y
\end{cases}
$$

Portanto, precisamos de duas rotinas básicas: substituição progressiva e substituição regressiva.

### Algoritmo 1: Resolução de sistemas triangulares inferiores

<div style="border:2px solid #444; padding:15px; border-radius:8px; background:#f7f7f7">

**Entrada**

- Matriz triangular inferior $L \in \mathbb{R}^{n \times n}$ com $\ell_{ii} \neq 0$, para todo $i$
- Vetor $b \in \mathbb{R}^n$

---

**Saída**

- Vetor solução $y \in \mathbb{R}^n$

---

**Passos**

1. Para $k = 1, 2, \ldots, n$, faça:

   - $s \leftarrow 0$
   - Para $j = 1, 2, \ldots, k-1$, faça:
      - $s \leftarrow s + \ell_{kj} y_j$
   - $y_k = \dfrac{b_k - s}{\ell_{kk}}$

---

📌 Observação:
O algoritmo realiza a chamada **substituição progressiva**, resolvendo o sistema de cima para baixo.

</div>

In [3]:
def subs_prog(L, b, verbose=True):
    """
    Resolve o sistema linear Ly = b, onde L é triangular inferior.
    """
    n = len(b)
    y = np.zeros(n)

    if verbose:
        imprimir_matriz(L, "Sistema triangular inferior")
        imprimir_vetor(b, "Vetor b")

    for k in range(n):
        s = 0.0
        for j in range(k):
            s += L[k, j] * y[j]

        y[k] = (b[k] - s) / L[k, k]

        if verbose:
            print(f"🔸 y{k+1} = {y[k]:.6e}")

    return y

### Algoritmo 2: Resolução de sistemas triangulares superiores

<div style="border:2px solid #444; padding:15px; border-radius:8px; background:#f7f7f7">

**Entrada**

- Matriz triangular superior $U \in \mathbb{R}^{n \times n}$ com $u_{ii} \neq 0$, para todo $i$
- Vetor $y \in \mathbb{R}^n$

---

**Saída**

- Vetor solução $x \in \mathbb{R}^n$

---

**Passos**

1. $x_n = \dfrac{y_n}{u_{nn}}$

2. Para $k = n-1, n-2, \ldots, 1$, faça:

   - $s \leftarrow 0$
   - Para $j = k+1, \ldots, n$, faça:
      - $s \leftarrow s + u_{kj}x_j$
   - $x_k = \dfrac{y_k - s}{u_{kk}}$

---

📌 Observação:
O algoritmo realiza a chamada **substituição regressiva**, resolvendo o sistema de baixo para cima.

</div>

In [4]:
def subs_reg(U, y, verbose=True):
    """
    Resolve o sistema linear Ux = y, onde U é triangular superior.
    """
    n = len(y)
    x = np.zeros(n)

    if verbose:
        imprimir_matriz(U, "Sistema triangular superior")
        imprimir_vetor(y, "Vetor y")

    x[n-1] = y[n-1] / U[n-1, n-1]

    if verbose:
        print(f"\n🔸 x{n} = {x[n-1]:.6e}")

    for k in range(n-2, -1, -1):
        s = 0.0
        for j in range(k+1, n):
            s += U[k, j] * x[j]

        x[k] = (y[k] - s) / U[k, k]

        if verbose:
            print(f"🔸 x{k+1} = {x[k]:.6e}")

    return x

**O que significa `verbose=True`?**

O parâmetro `verbose=True` indica que a função deve exibir informações adicionais durante a execução. Assim, conseguimos acompanhar:

- as matrizes usadas;
- os vetores intermediários;
- os valores calculados em cada etapa.

Se você usar `verbose=False`, a função executa normalmente, mas sem mostrar esses detalhes.

## $ \S 3 $ Cálculo dos fatores $L$ e $U$

A decomposição de Doolittle é a forma mais usual da fatoração LU em cursos introdutórios de Cálculo Numérico.

Nesse caso, impomos que a diagonal de $L$ seja composta apenas por 1. Assim, cada multiplicador usado na eliminação é armazenado abaixo da diagonal principal de $L$.

<div style="background-color: #D1FFBD; padding: 12px; border-radius: 6px;">

💡 Em linguagem de eliminação de Gauss: a matriz $U$ guarda o sistema escalonado, e a matriz $L$ guarda o histórico das eliminações.
</div>

### Algoritmo 3: Fatoração LU sem pivoteamento

<div style="border:2px solid #444; padding:15px; border-radius:8px; background:#f7f7f7">

**Entrada**

- Matriz $A \in \mathbb{R}^{n \times n}$

---

**Saída**

- Matrizes $L$ e $U$ tais que $A = LU$

---

**Passos**

1. Faça $L \leftarrow I$ e $U \leftarrow A$

2. Para $k = 1, 2, \ldots, n-1$, faça:

   - Para $i = k+1, \ldots, n$, faça:

      - $m_{ik} = \dfrac{u_{ik}}{u_{kk}}$
      - armazene $m_{ik}$ em $L$
      - substitua a linha $i$ de $U$ por:

      $$
      L_i \leftarrow L_i - m_{ik}L_k
      $$

---

📌 Observação:
Esse algoritmo só funciona quando os pivôs usados no processo são não nulos. Em particular, isso exige que o escalonamento possa ser feito sem troca de linhas.

</div>

In [5]:
def fatoracao_lu(A, verbose=True):
    """
    Calcula a fatoração LU da matriz A sem pivoteamento.
    """
    A = A.astype(float).copy()
    n = A.shape[0]

    L = np.eye(n)
    U = A.copy()

    if verbose:
        imprimir_matriz(A, "Matriz inicial A")

    for k in range(n-1):
        if np.isclose(U[k, k], 0.0):
            raise ValueError("Pivô nulo encontrado. A fatoração LU sem pivoteamento não pode continuar.")

        for i in range(k+1, n):
            m = U[i, k] / U[k, k]
            L[i, k] = m
            U[i, k:] = U[i, k:] - m * U[k, k:]

            if verbose:
                print(f"\n🔸 Eliminando o elemento a({i+1},{k+1})")
                print(f"m({i+1},{k+1}) = {m:.6e}")
                imprimir_fatoracao(L, U, titulo=f"Etapa {k+1}")

    return L, U

**🔹 Função auxiliar: resolução de sistemas por LU**

Depois de obtermos $A=LU$, podemos encapsular todo o processo em uma única função.

In [6]:
def resolver_lu(A, b, verbose=True):
    """
    Resolve Ax = b usando fatoração LU.
    """
    L, U = fatoracao_lu(A, verbose)

    if verbose:
        print("\n🔹 Resolvendo Ly = b")
    y = subs_prog(L, b, verbose)

    if verbose:
        print("\n🔹 Resolvendo Ux = y")
    x = subs_reg(U, y, verbose)

    return x, L, U, y

🧪[__Exemplo 3-02.1__](#exemplo-3-02.1) Considere a matriz:

$$
A = \begin{bmatrix}
3 & 2 & 4 \\
1 & 1 & 2 \\
4 & 3 & 2
\end{bmatrix}
$$

Encontre a fatoração $A = LU$.

In [7]:
A = np.array([[3.0, 2, 4],
              [1.0, 1, 2],
              [4.0, 3, 2]])

▶️ Resolução

Utilizando a função `fatoracao_lu` definida anteriormente:

In [8]:
L, U = fatoracao_lu(A)
print("\nResultado final:")
imprimir_matriz(L, "Matriz L")
imprimir_matriz(U, "Matriz U")


🔹 Matriz inicial A
   ┌                                    ┐
L1  3.000e+00  2.000e+00  4.000e+00 
L2  1.000e+00  1.000e+00  2.000e+00 
L3  4.000e+00  3.000e+00  2.000e+00 
   └                                    ┘

🔸 Eliminando o elemento a(2,1)
m(2,1) = 3.333333e-01

🔹 Etapa 1: matriz L
   ┌                                    ┐
L1  1.000e+00  0.000e+00  0.000e+00 
L2  3.333e-01  1.000e+00  0.000e+00 
L3  0.000e+00  0.000e+00  1.000e+00 
   └                                    ┘

🔹 Etapa 1: matriz U
   ┌                                    ┐
L1  3.000e+00  2.000e+00  4.000e+00 
L2  0.000e+00  3.333e-01  6.667e-01 
L3  4.000e+00  3.000e+00  2.000e+00 
   └                                    ┘

🔸 Eliminando o elemento a(3,1)
m(3,1) = 1.333333e+00

🔹 Etapa 1: matriz L
   ┌                                    ┐
L1  1.000e+00  0.000e+00  0.000e+00 
L2  3.333e-01  1.000e+00  0.000e+00 
L3  1.333e+00  0.000e+00  1.000e+00 
   └                                    ┘

🔹 Etapa 1: matriz U
   ┌    

#### 🔍 Verificação da fatoração

Para verificar a decomposição, basta calcular `L @ U`.

In [9]:
print("L @ U =\n", L @ U)
print("\nA original =\n", A)

L @ U =
 [[3. 2. 4.]
 [1. 1. 2.]
 [4. 3. 2.]]

A original =
 [[3. 2. 4.]
 [1. 1. 2.]
 [4. 3. 2.]]


🧪[__Exemplo 3-02.2__](#exemplo-3-02.2) Considere o sistema:

$$
\left\{
\begin{aligned}
3x_1 + 2x_2 + 4x_3 &= 1 \\
x_1 + x_2 + 2x_3 &= 2 \\
4x_1 + 3x_2 + 2x_3 &= 3
\end{aligned}
\right.
$$

Resolva o sistema utilizando fatoração LU.

In [10]:
A = np.array([[3.0, 2, 4],
              [1.0, 1, 2],
              [4.0, 3, 2]])

b = np.array([1.0, 2, 3])

▶️ Resolução

Utilizando a função `resolver_lu`:

In [11]:
x, L, U, y = resolver_lu(A, b)

print("\nSolução:", x)


🔹 Matriz inicial A
   ┌                                    ┐
L1  3.000e+00  2.000e+00  4.000e+00 
L2  1.000e+00  1.000e+00  2.000e+00 
L3  4.000e+00  3.000e+00  2.000e+00 
   └                                    ┘

🔸 Eliminando o elemento a(2,1)
m(2,1) = 3.333333e-01

🔹 Etapa 1: matriz L
   ┌                                    ┐
L1  1.000e+00  0.000e+00  0.000e+00 
L2  3.333e-01  1.000e+00  0.000e+00 
L3  0.000e+00  0.000e+00  1.000e+00 
   └                                    ┘

🔹 Etapa 1: matriz U
   ┌                                    ┐
L1  3.000e+00  2.000e+00  4.000e+00 
L2  0.000e+00  3.333e-01  6.667e-01 
L3  4.000e+00  3.000e+00  2.000e+00 
   └                                    ┘

🔸 Eliminando o elemento a(3,1)
m(3,1) = 1.333333e+00

🔹 Etapa 1: matriz L
   ┌                                    ┐
L1  1.000e+00  0.000e+00  0.000e+00 
L2  3.333e-01  1.000e+00  0.000e+00 
L3  1.333e+00  0.000e+00  1.000e+00 
   └                                    ┘

🔹 Etapa 1: matriz U
   ┌    

#### 🔍 Verificação da solução

Após resolver o sistema, é importante verificar se a solução obtida está correta.

In [12]:
print("A @ x =", A @ x)
print("b     =", b)

A @ x = [1. 2. 3.]
b     = [1. 2. 3.]


## $ \S 4 $ Vantagem quando a matriz é fixa

Uma das principais vantagens da fatoração LU, aparece quando precisamos resolver vários sistemas com a mesma matriz $A$ e vetores independentes diferentes.

🧪[__Exemplo 3-02.3__](#exemplo-3-02.3) Quando a matriz é a mesma, a fatoração pode ser reaproveitada.

Considere:

$$
A = \begin{bmatrix}
2 & 1 & 1 \\
4 & 3 & 3 \\
8 & 7 & 9
\end{bmatrix},
\quad
b_1 = \begin{bmatrix}5 \\ 11 \\ 29\end{bmatrix},
\quad
b_2 = \begin{bmatrix}1 \\ 2 \\ 3\end{bmatrix}.
$$

In [13]:
A = np.array([[2.0, 1, 1],
              [4.0, 3, 3],
              [8.0, 7, 9]])

b1 = np.array([5.0, 11, 29])
b2 = np.array([1.0, 2, 3])

##Terminar a fatoração LU de A e resolver os sistemas Ax = b1 e Ax = b2 usando as funções implementadas.

<div style="background-color: #D1FFBD; padding: 12px; border-radius: 6px;">

💡 Quando a matriz $A$ é fixa, a etapa mais cara é feita apenas uma vez. Depois disso, basta resolver dois sistemas triangulares para cada novo vetor $b$.
</div>

## $ \S 5 $ Fatoração com pivoteamento

Pré-multiplicar $A$ por uma matriz de permutação $P$ é equivalente a trocar as linhas de $A$. Considere o caso em que $P$ é a matriz de permutação que troca as linhas 1 e 2. Então,

$$
PA = \begin{bmatrix}0 & 1 & 0 \\
1 & 0 & 0 \\
0 & 0 & 1\end{bmatrix}
\begin{bmatrix}3 & 2 & 4 \\
4 & 3 & 3 \\
8 & 7 & 9\end{bmatrix}
=
\begin{bmatrix}4 & 3 & 3 \\
3 & 2 & 4 \\
8 & 7 & 9\end{bmatrix}
$$

Verifique executando as celulas abaixo:

In [14]:
P = np.array([[0, 1, 0],
              [1, 0, 0],
              [0, 0, 1]])
imprimir_matriz(P, "Matriz de permutação P")


🔹 Matriz de permutação P
   ┌                                    ┐
L1  0.000e+00  1.000e+00  0.000e+00 
L2  1.000e+00  0.000e+00  0.000e+00 
L3  0.000e+00  0.000e+00  1.000e+00 
   └                                    ┘


In [15]:
A = np.array([[3.0, 2, 4],
              [4, 3, 3],
              [8, 7, 9]])
imprimir_matriz(A, "Matriz A")


🔹 Matriz A
   ┌                                    ┐
L1  3.000e+00  2.000e+00  4.000e+00 
L2  4.000e+00  3.000e+00  3.000e+00 
L3  8.000e+00  7.000e+00  9.000e+00 
   └                                    ┘


In [16]:
imprimir_matriz(P @ A, "PA")



🔹 PA
   ┌                                    ┐
L1  4.000e+00  3.000e+00  3.000e+00 
L2  3.000e+00  2.000e+00  4.000e+00 
L3  8.000e+00  7.000e+00  9.000e+00 
   └                                    ┘



A não singularidade de $A$ não garante, por si só, que a fatoração $A = LU$ exista sem troca de linhas.

Quando um pivô é nulo, ou muito pequeno, precisamos realizar trocas de linhas. Nesse caso, trabalhamos com a fatoração

$$
PA = LU,
$$

em que:

- $P$ é uma matriz de permutação;
- $L$ é triangular inferior com diagonal igual a 1;
- $U$ é triangular superior.

A seguir, vamos construir uma rotina simples com **pivoteamento parcial**.

### Algoritmo 4: Fatoração LU com pivoteamento parcial (forma computacional)

<div style="border:2px solid #444; padding:15px; border-radius:8px; background:#f7f7f7">

**Entrada**

- Matriz $A \in \mathbb{R}^{n \times n}$

---

**Saída**

- Matriz $A$ modificada contendo $L$ e $U$
- Vetor de permutação $p \in \mathbb{R}^n$

tais que $PA = LU$

---

**Passos**

1. **Inicialização:**

- Para $i = 1, \ldots, n$, faça:
  
  $p(i) \leftarrow i$

---

2. **Para $k = 1, \ldots, n-1$, faça:**

🔸 **Escolha do pivô**

- $pv \leftarrow |a_{kk}|$
- $r \leftarrow k$

- Para $i = k+1, \ldots, n$, faça:

  - Se $|a_{ik}| > pv$, então:
    - $pv \leftarrow |a_{ik}|$
    - $r \leftarrow i$

- Se $pv = 0$, então:

  ⚠️ **Parar** (a matriz é singular)

---

🔸 **Troca de linhas**

- Se $r \neq k$, então:

  - trocar $p(k) \leftrightarrow p(r)$

  - Para $j = 1, \ldots, n$, faça:
  
    - trocar $a_{kj} \leftrightarrow a_{rj}$

---

🔸 **Eliminação**

- Para $i = k+1, \ldots, n$, faça:

  - $m \leftarrow \dfrac{a_{ik}}{a_{kk}}$
  - $a_{ik} \leftarrow m$  *(armazena o multiplicador em $L$)*

  - Para $j = k+1, \ldots, n$, faça:

    - $a_{ij} \leftarrow a_{ij} - m \, a_{kj}$

---

📌 **Observações:**

- A matriz final contém:
  - $U$ na parte superior (incluindo a diagonal)
  - $L$ abaixo da diagonal (com diagonal unitária implícita)

- O vetor $p$ representa as permutações realizadas

- Este formato é mais eficiente computacionalmente, pois evita armazenar $L$ e $U$ separadamente

</div>

In [17]:
def lu_pivoteamento(A, verbose=False):
    """
    Fatoração LU com pivoteamento parcial, seguindo Ruggiero e Lopes.
    Retorna:
        P -> matriz de permutação
        L -> matriz triangular inferior com diagonal 1
        U -> matriz triangular superior
    """
    A = A.astype(float).copy()
    n = A.shape[0]

    U = A.copy()
    L = np.eye(n)
    P = np.eye(n)

    if verbose:
        imprimir_matriz(A, "Matriz inicial A")

    for k in range(n-1):
        # 🔹 Escolha do pivô
        coluna = np.abs(U[k:, k])
        r = np.argmax(coluna) + k

        if verbose:
            print(f"\n🔸 Etapa {k+1}: Coluna {k+1}")
            print(f"Pivô candidato: linha {r+1}, valor {U[r, k]:.6e}")

        if np.isclose(U[r, k], 0.0):
            raise ValueError("Matriz singular.")

        # 🔹 Troca de linhas em U e P
        if r != k:
            U[[k, r]] = U[[r, k]]
            P[[k, r]] = P[[r, k]]
            # Troca em L as colunas já calculadas (0 a k-1)
            L[[k, r], :k] = L[[r, k], :k]

            if verbose:
                print(f"Troca de linhas {k+1} e {r+1}")
                imprimir_fatoracao_pivotada(P, L, U, f"Após troca na etapa {k+1}")

        # 🔹 Eliminação
        for i in range(k+1, n):
            m = U[i, k] / U[k, k]
            L[i, k] = m
            U[i, k:] = U[i, k:] - m * U[k, k:]

            if verbose:
                print(f"Eliminação: L[{i+1},{k+1}] = {m:.6e}")
                imprimir_fatoracao_pivotada(P, L, U, f"Após eliminação na linha {i+1}, etapa {k+1}")

    return P, L, U

**🔹 Função auxiliar: resolução de sistemas com pivoteamento**

Se $PA = LU$, então para resolver $Ax=b$ resolvemos primeiro

$$
Ly = Pb
$$

e depois

$$
Ux = y.
$$

In [18]:
def resolver_palu(P, L, U, b, verbose=False):
    """
    Resolve Ax = b usando a fatoração PA = LU.
    """

    n = len(b)

    # 🔹 Aplica permutação: c = Pb
    c = P @ b
    
    if verbose:
        imprimir_vetor(b, "Vetor b")
        imprimir_matriz(P, "Matriz P")
        imprimir_vetor(c, "Vetor c = P b")

    # 🔹 Resolve Ly = c (substituição progressiva)
    if verbose:
        print("\n🔹 Resolvendo Ly = c")
    y = subs_prog(L, c, verbose)

    # 🔹 Resolve Ux = y (substituição regressiva)
    if verbose:
        print("\n🔹 Resolvendo Ux = y")
    x = subs_reg(U, y, verbose)

    return x

In [19]:
def metodo_lu_pivoteamento(A, b, verbose=False):
    """
    Resolve Ax = b utilizando fatoração LU com pivoteamento parcial.
    
    Parâmetros:
        A: matriz dos coeficientes (n x n)
        b: vetor dos termos independentes (n,)
        verbose: se True, imprime os passos da fatoração e resolução
    
    Retorna:
        x: solução do sistema
    """
    P, L, U = lu_pivoteamento(A, verbose)
    x = resolver_palu(P, L, U, b, verbose)
    return x

🧪[__Exemplo 3-02.4__](#exemplo-3-02.4) Considere a matriz

$$
A = \begin{bmatrix}
0 & 1 \\
2 & 3
\end{bmatrix}
$$

Essa matriz é inversível, mas o primeiro pivô é nulo. Vamos obter uma fatoração com pivoteamento.

In [20]:
A = np.array([[0.0, 1],
              [2.0, 3]])

P, L, U = lu_pivoteamento(A)

print("\nVerificação de PA = LU")
print("P @ A =\n", P @ A)
print("\nL @ U =\n", L @ U)


Verificação de PA = LU
P @ A =
 [[2. 3.]
 [0. 1.]]

L @ U =
 [[2. 3.]
 [0. 1.]]


🧪[__Exemplo 3-02.5__](#exemplo-3-02.5) Resolva o sistema

$$
\left\{
\begin{aligned}
10^{-20}x_1 + x_2 &= 1 \\
x_1 + x_2 &= 2
\end{aligned}
\right.
$$

utilizando a fatoração com pivoteamento.

In [21]:
A = np.array([[1e-20, 1.0],
              [1.0,   1.0]])

b = np.array([1.0, 2.0])

x = metodo_lu_pivoteamento(A, b)
print("\nSolução:", x)
print("Verificação:", A @ x)


Solução: [1. 1.]
Verificação: [1. 2.]


Verifique a execução utilizando a fatoração sem pivoteamento para comparar os resultados.

In [22]:
A = np.array([[0.0, 1],
              [2.0, 3]])

try:
    L, U = fatoracao_lu(A)
except ValueError as erro:
    print("Erro:", erro)


🔹 Matriz inicial A
   ┌                        ┐
L1  0.000e+00  1.000e+00 
L2  2.000e+00  3.000e+00 
   └                        ┘
Erro: Pivô nulo encontrado. A fatoração LU sem pivoteamento não pode continuar.


<div style="background-color:#ffcccc; padding:12px; border-radius:6px;">

🚨 Sem pivoteamento, a fatoração LU pode falhar mesmo quando a matriz é inversível.

</div>

📝[__Exercício 3-02.1__](#exercicio-3-02.1): Resolva o sistema

$$
\left\{
\begin{aligned}
3x_1 - 4x_2 + x_3 &= 9 \\
x_1 + 2x_2 + 2x_3 &= 3 \\
4x_1 \quad\quad\, - 3x_3 &= -2
\end{aligned}
\right.
$$

utilizando a fatoração com pivoteamento.

📝[__Exercício 3-02.1__](#exercicio-3-02.1): Encontre a fatoração $A = LU$ da matriz

$$
A = \begin{bmatrix}
1 & 2 & 1 \\
2 & 5 & 4 \\
3 & 8 & 6
\end{bmatrix}.
$$

_Solução:_

📝[__Exercício 3-02.2__](#exercicio-3-02.2): Resolva o sistema abaixo utilizando fatoração LU:

$$
\left\{
\begin{aligned}
x_1 + 2x_2 + x_3 &= 4 \\
2x_1 + 5x_2 + 4x_3 &= 3 \\
3x_1 + 8x_2 + 6x_3 &= 5
\end{aligned}
\right.
$$


_Solução:_

📝[__Exercício 3-02.3__](#exercicio-3-02.3): Utilize pivoteamento parcial para obter uma fatoração $PA = LU$ da matriz

$$
A = \begin{bmatrix}
0 & 2 & 1 \\
1 & 3 & 4 \\
2 & 1 & 3
\end{bmatrix}.
$$

<div style="background-color: #ffffbd; padding: 12px; border-radius: 6px;">

💡 Na prática computacional, a fatoração $PA = LU$ é geralmente preferida, pois incorpora o pivoteamento e torna o processo mais robusto.
</div>

📝[__Exercício 3-02.4__](#exercicio-3-02.4):
<div style="background-color: #f5f5f5; padding: 18px; border-radius: 8px;">

<div style="display: flex; align-items: center; gap: 20px;">

<div style="flex: 1;">
<img src="figures/fab_comp.png" width="300">
<p style="text-align: center; font-size: 12px;">
Figura ilustrativa
</p>
</div>

<div style="flex: 2;">

Um engenheiro de produção supervisiona a fabricação de **quatro tipos de computadores** 🖥️.  

Para a produção, são necessários quatro tipos de recursos:

- mão de obra  
- metais  
- plásticos  
- componentes eletrônicos  

As quantidades de recursos necessárias para produzir cada tipo de computador são dadas pela tabela:

$$
\begin{array}{c|cccc}
\text{Tipo} & \text{Mão de obra (h)} & \text{Metais (kg)} & \text{Plásticos (kg)} & \text{Componentes} \\
\hline
1 & 3 & 20 & 10 & 10 \\
2 & 4 & 25 & 15 & 8 \\
3 & 7 & 40 & 20 & 10 \\
4 & 20 & 50 & 22 & 15 \\
\end{array}
$$

---

Considere o consumo diário disponível de recursos:

- $504$ horas de mão de obra  
- $1970$ kg de metais  
- $970$ kg de plásticos  
- $601$ componentes  

a) Utilize um **método direto e estável** para determinar o número de computadores de cada tipo produzidos por dia.

---

📌 **Observação:**  
A solução deve fornecer o número de computadores de cada tipo (valores inteiros).

</div>

</div>
</div>

_Solução:_

### ⚠️ Erros comuns

- Não aplicar pivoteamento quando necessário
- Montar incorretamente a matriz $L$
- Esquecer de permutar o vetor $b$
- Erros na substituição retroativa

---

## 📚 Referências

> 📘 RUGGIERO, Márcia A. Gomes; LOPES, Vera Lúcia da Rocha.  
> **Cálculo numérico: aspectos teóricos e computacionais**.  
> 2. ed. São Paulo: Makron Books, 1996.